# Train LocalExplorerAI Experience Ranker (verified human labels)

Notebook này chỉ nhận dữ liệu nhãn người thật đã được rà soát, gắn với POI/trải nghiệm/slot có nguồn xác minh và kết quả planner. File template hiện chỉ có header, không chứa dữ liệu giả. Không dùng `ranking_candidates_labeled.csv`, OSM crawl chưa xác minh, nhãn weak/heuristic/LLM, email/chat thô hoặc thông tin định danh khách.

Chỉ xuất `local_explorer_ranker_bundle.zip` nếu model ML vượt rule baseline ít nhất 0.01 NDCG@5 trên validation và không kém baseline trên test chưa đụng tới. Nếu không đạt, notebook sẽ không cho xuất ZIP deploy được; API tiếp tục dùng `rule-baseline-v1`.

## 1. Chuẩn bị dữ liệu

Tải [`experience_ranker_verified_template.csv`](https://github.com/NguyenTriBaoThang/LocalExplorerAI/blob/main/data/training/experience_ranker_verified_template.csv), điền nhiều dòng ứng viên cho mỗi `query_id`, rồi lưu CSV UTF-8 vào Google Drive tại `MyDrive/LocalExplorerAI/dataset/training/experience_ranker_verified.csv`. Mỗi truy vấn cần ít nhất hai ứng viên khả thi và có đánh giá 0–3 do người thật gán theo cùng rubric: **3** đạt mục đích, đúng cam kết; **2** phương án thay thế giữ được bản chất, chỉ đánh đổi nhỏ; **1** lệch mục đích (ví dụ khách muốn tự làm nhưng chỉ được ngắm); **0** thất bại thực tế nghiêm trọng. Dùng mã reviewer giả danh (không phải tên/email). `estimated_columns` để trống; không ghi `training_eligible=true` nếu thiếu bằng chứng.

Cột bắt buộc bao gồm địa chỉ/tọa độ và nguồn POI, nguồn trải nghiệm/giá/thời lượng, slot đang mở và sức chứa đủ nhóm, snapshot thời gian, ETA đi-về, thời hạn về, bằng chứng planner, timestamp có múi giờ và hạn dùng bằng chứng, kết quả chuyến đi đã hoàn tất và nguồn nhãn. Script sẽ dừng nếu thiếu trường hoặc provenance. `labeler_ref` chỉ là mã giả danh, tuyệt đối không ghi tên/email.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/LocalExplorerAI')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/NguyenTriBaoThang/LocalExplorerAI.git', str(REPO)], check=True)
print('Repository:', REPO)

In [ ]:
%cd /content/LocalExplorerAI
%pip install -r apps/api/requirements-ranking-training.txt

## 2. Kiểm tra CSV và chạy benchmark công bằng

Train/test được chia theo `query_id`; train và inference cùng gọi feature builder trong API. ID, nhãn, lý do gán nhãn và provenance không được dùng làm feature. Test chỉ được chấm sau khi validation chọn model.

In [ ]:
from datetime import datetime, timezone
import json

DATA = Path('/content/drive/MyDrive/LocalExplorerAI/dataset/training/experience_ranker_verified.csv')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/LocalExplorerAI/ranker_runs') / RUN_ID
if not DATA.is_file():
    raise FileNotFoundError(f'Không tìm thấy CSV dữ liệu thật: {DATA}. Hãy tải template, điền dữ liệu đã xác minh và lưu đúng vị trí.')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
command = ['python', 'scripts/train_verified_experience_ranker.py', '--data', str(DATA), '--output', str(OUTPUT)]
print('Command:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
report = json.loads((OUTPUT / 'benchmark_metrics.json').read_text(encoding='utf-8'))
print('Selected:', report['selected_model'])
print('Deployment eligible:', report['deployment_eligible'])
print('Validation baseline NDCG@5:', report['validation_baseline']['NDCG@5'])
print('Best ML validation NDCG@5:', report['best_ml_validation']['NDCG@5'])
print('Bundle:', report['bundle'])

## 3. Chỉ tải model nếu đủ điều kiện

Nếu validation/test không đạt cổng chất lượng, kết quả chỉ là benchmark nghiên cứu và không được cài vào production. Hãy xem `model_card.md`, `validation_comparison.csv`, `test_metrics.json` trước khi đưa model vào dự án.

In [ ]:
from google.colab import files

if report['deployment_eligible'] and report['bundle']:
    bundle_path = Path(report['bundle'])
    if not bundle_path.is_file() or bundle_path.name != 'local_explorer_ranker_bundle.zip':
        raise RuntimeError('Bundle path/name không đúng hợp đồng deploy.')
    files.download(str(bundle_path))
else:
    print('Không xuất bundle deploy được. Không đổi metadata thủ công; API vẫn dùng rule-baseline-v1.')
    print('Artifacts nghiên cứu được lưu tại:', OUTPUT)

## 4. Cài ZIP vào dự án

Giải nén `local_explorer_ranker_bundle.zip` trực tiếp vào `models/ai-ranker/` sao cho `model_metadata.json`, `feature_schema.json`, `preprocessing.json` và file model nằm ngay trong thư mục đó (không tạo thêm thư mục con). Giữ `RANKER_MODEL_DIR=/models/ai-ranker` trong Docker hoặc `RANKER_MODEL_DIR=./models/ai-ranker` khi chạy API trực tiếp. Cài runtime `apps/api/requirements-ml.txt`, khởi động lại API, gọi thử một request ranking hợp lệ để model được nạp, rồi kiểm tra `GET /api/ml/status`: cần `active_ranker=ml_ranker`, `deployment_allowed=true`, `loaded=true`. Nếu chưa, xem `block_reason`/`error`; fallback baseline vẫn an toàn. Không sửa `deployment_eligible` để ép nạp model.